# S-Fig 6 — Modality Ablation Bar Chart

ΔAUROC per ablation condition (No BAS, No RESP, etc.) relative to fast-ch baseline.  
**Source**: phase0_v3_abl, phase0_v3, phase0_v3_full  
**Tasks**: main tasks (ablation only covers these)  
**Head**: LSTM

In [ ]:
import sys
from pathlib import Path

# ── Workspace root (parent of NSRR-tools/) ────────────────────────────────────
WORKSPACE_ROOT = Path("../../../../..").resolve()   # adjust if notebook depth differs
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path
sys.path.insert(0, str(Path(".").resolve()))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib
matplotlib.use("Agg")   # comment out in Jupyter to get inline plots
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()
print("Setup OK — workspace root:", WORKSPACE_ROOT)

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
HEAD   = "lstm"
SPLIT  = "test"
TASKS  = MAIN_TASKS    # ablation only covers main tasks

# For ablation, do NOT filter by k — load raw then pass to function
import pandas as pd
from pathlib import Path

def _load_raw(exp):
    p = WORKSPACE_ROOT / "final_results" / exp / "collected" / "analysis.csv"
    df = pd.read_csv(p)
    if "context_length_min" not in df.columns:
        df["context_length_min"] = df["context_length"].map(
            {"30s": 0.5, "10m": 10.0, "40m": 40.0,
             "80m": 80.0, "120m": 120.0, "240m": 240.0}.get)
    return df

df_abl  = _load_raw("phase0_v3_abl")
df_fast = _load_raw("phase0_v3")
df_full = _load_raw("phase0_v3_full")

print("Ablation run_tags:", sorted(df_abl["run_tag"].unique()))

In [ ]:
N_COLS = 5   # one column per task side-by-side (classic modality bar layout)
fig, axes = plt.subplots(1, N_COLS, figsize=(FULL_W, 2.8), sharey=False)

for col, (ax, task) in enumerate(zip(axes, TASKS)):
    panels.modality_bar_panel(ax, df_abl, df_fast, df_full, task, head=HEAD, split=SPLIT)
    ax.set_title(TASK_LABEL[task], fontsize=8)
    add_panel_label(ax, f"({chr(97+col)})")
    if col > 0:
        ax.set_yticklabels([])

fig.tight_layout(w_pad=0.5)
save_figure(fig, FINAL_OUT, "sfig6_modality_ablation")
plt.show()